# SAM3 x iNaturalist - Colab pipeline

Taxon ID -> iNat metadata -> photo download -> SAM3 text-prompted segmentation -> images, masks (`.npy`), overlays, transparent segment cut-outs, and per-mask average RGB.
Edit the **Config** cell, then *Runtime > Run all*. Needs a **GPU runtime** and a Hugging Face token with access to `facebook/sam3`.

In [ ]:
# ============================================================
# CONFIG  -- everything that used to be a command-line argument
# ============================================================

# -- What to segment ------------------------------------------------
TAXON_ID    = 62741        # iNaturalist taxon ID
TEXT_PROMPT = "petal"      # SAM3 text prompt
YEAR_MIN    = 2020         # observed-on year range, inclusive
YEAR_MAX    = 2024
QUALITY     = "research"   # iNat quality grade

# -- Test run -------------------------------------------------------
# TEST_RUN caps the METADATA pull at N observations. An observation can carry
# several photos, so expect somewhat more than N images. It writes to its own
# "..._testN" folder, so a test can never mix into a full run.
TEST_RUN            = True
TEST_N_OBSERVATIONS = 50

# -- Thresholds -----------------------------------------------------
CONF_THRESHOLD = 0.9       # keep only detections scoring >= this
MASK_THRESHOLD = 0.8       # binarisation threshold for the mask logits

# -- Google Drive output --------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/sam3_runs"
RUN_TAG    = ""            # optional suffix; change it to force a brand-new folder

# -- Which artifacts to save ----------------------------------------
SAVE_IMAGES   = True       # the downloaded original photo
SAVE_MASKS    = True       # one .npy per detection
SAVE_OVERLAYS = True       # annotated image (mask tint + bbox + score label)
SAVE_SEGMENTS = True       # transparent PNG cut-out per detection

# -- Performance ----------------------------------------------------
MAX_WORKERS   = 8          # parallel photo downloads (this pipeline is download-bound)
BATCH_SIZE    = 1000       # images per batch_NNNNN subfolder
PER_PAGE      = 200        # iNat API page size (200 is the max)
API_DELAY_SEC = 1.1        # politeness delay between metadata pages
TIMEOUT_SEC   = 60

# -- Resume ---------------------------------------------------------
# Both stages resume: metadata picks up from the last observation_id in the CSV,
# segmentation skips any global_index already logged in mask_summary.csv.
RESUME = True

### Setup
Installs pinned deps, authenticates to Hugging Face (add `HF_TOKEN` under the Colab key icon, or paste it when prompted), and mounts Drive.

In [ ]:
!pip install -q transformers==5.8.1 accelerate==1.13.0 "huggingface_hub>=1.0" 2>&1 | tail -2

import os
from getpass import getpass

# -- Hugging Face auth (facebook/sam3 is a gated repo) ---------------
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass("HF token (needs access to facebook/sam3): ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"]  = "/content/hf_cache"   # model cache on local disk, not Drive

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

# -- Google Drive ----------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    print("\n!! No GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

In [ ]:
# ============================================================
# Imports and derived output paths
# ============================================================
import csv, json, re, time
from io import BytesIO
from typing import Any, Dict, Optional
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from transformers import Sam3Model, Sam3Processor

Image.MAX_IMAGE_PIXELS = None   # iNat originals can be very large

# -- Run folder ------------------------------------------------------
_prompt_slug = re.sub(r"[^A-Za-z0-9]+", "_", TEXT_PROMPT).strip("_").lower()
RUN_NAME = f"taxon{TAXON_ID}_{_prompt_slug}_{YEAR_MIN}-{YEAR_MAX}_conf{CONF_THRESHOLD}"
if TEST_RUN:
    RUN_NAME += f"_test{TEST_N_OBSERVATIONS}"
if RUN_TAG:
    RUN_NAME += f"_{RUN_TAG}"

BASE_DIR     = os.path.join(DRIVE_ROOT, RUN_NAME)
IMAGE_BASE   = os.path.join(BASE_DIR, "images")
MASK_BASE    = os.path.join(BASE_DIR, "masks")
OVERLAY_BASE = os.path.join(BASE_DIR, "overlays")
SEGMENT_BASE = os.path.join(BASE_DIR, "segments")

METADATA_CSV = os.path.join(BASE_DIR, f"inat_taxon_{TAXON_ID}_{QUALITY}_metadata.csv")
OUT_CSV      = os.path.join(BASE_DIR, "mask_summary.csv")
ERR_LOG      = os.path.join(BASE_DIR, "errors.txt")
TIMING_JSON  = os.path.join(BASE_DIR, "run_summary.json")

for d in [BASE_DIR, IMAGE_BASE, MASK_BASE, OVERLAY_BASE, SEGMENT_BASE]:
    os.makedirs(d, exist_ok=True)

# -- Fixed constants -------------------------------------------------
API_BASE       = "https://api.inaturalist.org/v1/observations"
CHUNK_BYTES    = 1024 * 1024
SKIP_IF_EXISTS = True
OVERLAY_ALPHA, OVERLAY_COLOR      = 95, (255, 0, 0)
BOX_COLOR, BOX_WIDTH, TEXT_MARGIN = (255, 255, 0), 4, 4

print("Output folder:", BASE_DIR)
print("Prompt       :", repr(TEXT_PROMPT), "| conf >=", CONF_THRESHOLD,
      "| years", f"{YEAR_MIN}-{YEAR_MAX}")
print("Test run     :", TEST_RUN,
      f"({TEST_N_OBSERVATIONS} observations)" if TEST_RUN else "(full taxon)")

In [ ]:
# ============================================================
# HTTP helpers
# ============================================================
def make_session(max_workers: int = 8) -> requests.Session:
    s = requests.Session()
    retry = Retry(total=8, backoff_factor=1.5,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",), raise_on_status=False)
    adapter = HTTPAdapter(max_retries=retry,
                          pool_connections=max_workers, pool_maxsize=max_workers)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": "inat-sam3-colab/1.0", "Accept": "application/json"})
    return s


def fetch_json(session, url, params, timeout=None) -> Dict[str, Any]:
    timeout = TIMEOUT_SEC if timeout is None else timeout
    r = session.get(url, params=params, timeout=timeout)
    if r.status_code == 429:
        time.sleep(10)
        r = session.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()


def download_image_bytes(session, url) -> bytes:
    r = session.get(url, stream=True, timeout=TIMEOUT_SEC)
    if r.status_code == 429:
        time.sleep(10)
        r = session.get(url, stream=True, timeout=TIMEOUT_SEC)
    r.raise_for_status()
    return b"".join(c for c in r.iter_content(chunk_size=CHUNK_BYTES) if c)


def write_bytes_atomic(out_path, data):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    tmp = out_path + ".part"
    with open(tmp, "wb") as f:
        f.write(data)
    os.replace(tmp, out_path)

In [ ]:
# ============================================================
# Stage 1 - get_metadata(taxon_id): iNat API -> metadata CSV
# ============================================================
METADATA_COLUMNS = [
    "observation_id", "observation_uuid", "quality_grade", "observed_on",
    "time_observed_at", "created_at", "updated_at", "license_code", "geoprivacy",
    "taxon_geoprivacy", "location", "latitude", "longitude", "place_guess",
    "captive", "identifications_count", "comments_count", "faves_count",
    "user_id", "user_login", "taxon_id", "taxon_name",
    "taxon_preferred_common_name", "taxon_rank", "taxon_ancestry",
    "photo_id", "photo_license_code", "photo_attribution",
    "photo_width", "photo_height", "photo_url_original",
]


def _lat_lng(obs):
    """v1 returns 'location' as the string 'lat,lng'; there are no top-level
    latitude/longitude keys, so parse them out (obscured obs give None)."""
    loc = obs.get("location")
    if isinstance(loc, str) and "," in loc:
        try:
            lat, lng = loc.split(",", 1)
            return float(lat), float(lng)
        except ValueError:
            pass
    return obs.get("latitude"), obs.get("longitude")


def infer_ext_from_url(url: Optional[str]) -> Optional[str]:
    if not url:
        return None
    try:
        base = os.path.basename(urlparse(url).path)
        if "." not in base:
            return None
        ext = base.rsplit(".", 1)[1].lower()
        return ext if ext in {"jpg", "jpeg", "png", "gif"} else None
    except Exception:
        return None


def best_original_photo_url(photo: Dict[str, Any]) -> Optional[str]:
    """Prefer the open-data S3 original; fall back to whatever the API gave us."""
    pid = photo.get("id")
    api_url, api_original = photo.get("url"), photo.get("original_url")
    ext = infer_ext_from_url(api_original) or infer_ext_from_url(api_url)
    if pid and ext:
        return f"https://inaturalist-open-data.s3.amazonaws.com/photos/{pid}/original.{ext}"
    if api_original:
        return api_original
    if api_url:
        for token in ["square", "small", "medium", "large", "original"]:
            if f"/{token}." in api_url:
                return api_url.replace(f"/{token}.", "/original.")
        return api_url
    return None


def rows_from_obs(obs: Dict[str, Any]):
    taxon, user = obs.get("taxon") or {}, obs.get("user") or {}
    lat, lng = _lat_lng(obs)
    base = {
        "observation_id": obs.get("id"), "observation_uuid": obs.get("uuid"),
        "quality_grade": obs.get("quality_grade"), "observed_on": obs.get("observed_on"),
        "time_observed_at": obs.get("time_observed_at"), "created_at": obs.get("created_at"),
        "updated_at": obs.get("updated_at"), "license_code": obs.get("license_code"),
        "geoprivacy": obs.get("geoprivacy"), "taxon_geoprivacy": obs.get("taxon_geoprivacy"),
        "location": obs.get("location"), "latitude": lat,
        "longitude": lng, "place_guess": obs.get("place_guess"),
        "captive": obs.get("captive"),
        "identifications_count": obs.get("identifications_count"),
        "comments_count": obs.get("comments_count"), "faves_count": obs.get("faves_count"),
        "user_id": user.get("id"), "user_login": user.get("login"),
        "taxon_id": taxon.get("id"), "taxon_name": taxon.get("name"),
        "taxon_preferred_common_name": taxon.get("preferred_common_name"),
        "taxon_rank": taxon.get("rank"), "taxon_ancestry": taxon.get("ancestry"),
    }
    photos = obs.get("photos") or []
    if not photos:
        row = dict(base)
        for c in METADATA_COLUMNS:
            row.setdefault(c, "")
        yield row
        return
    for photo in photos:
        row = dict(base)
        dims = photo.get("original_dimensions") or {}
        row.update({
            "photo_id": photo.get("id"),
            "photo_license_code": photo.get("license_code"),
            "photo_attribution": photo.get("attribution"),
            "photo_width": photo.get("width") or dims.get("width"),
            "photo_height": photo.get("height") or dims.get("height"),
            "photo_url_original": best_original_photo_url(photo),
        })
        for c in METADATA_COLUMNS:
            row.setdefault(c, "")
        yield row


def _resume_last_obs_id(path) -> int:
    if not (RESUME and os.path.exists(path)):
        return 0
    with open(path, "rb") as f:
        try:
            f.seek(-65536, os.SEEK_END)
        except OSError:
            f.seek(0)
        lines = f.read().splitlines()
    last = next((l.decode("utf-8", "ignore") for l in reversed(lines) if l.strip()), None)
    if not last or last.startswith("observation_id,"):
        return 0
    try:
        return int(last.split(",", 1)[0].strip())
    except ValueError:
        return 0


def get_metadata(taxon_id, year_min=None, year_max=None, max_observations=None):
    """Page the iNat observations API and append one CSV row per photo."""
    year_min = YEAR_MIN if year_min is None else year_min
    year_max = YEAR_MAX if year_max is None else year_max

    session  = make_session(max_workers=8)
    id_above = _resume_last_obs_id(METADATA_CSV)
    print(f"Metadata resume: id_above={id_above}")

    existed = os.path.exists(METADATA_CSV) and os.path.getsize(METADATA_CSV) > 0
    out     = open(METADATA_CSV, "a", newline="", encoding="utf-8")
    writer  = csv.DictWriter(out, fieldnames=METADATA_COLUMNS)
    if not existed:
        writer.writeheader()

    params_base = {
        "taxon_id": taxon_id, "quality_grade": QUALITY, "per_page": PER_PAGE,
        "order": "asc", "order_by": "id", "photos": "true",
        "d1": f"{year_min}-01-01", "d2": f"{year_max}-12-31",
    }

    obs_count = row_count = 0
    try:
        while True:
            params = dict(params_base)
            if id_above > 0:
                params["id_above"] = id_above

            results = fetch_json(session, API_BASE, params).get("results", [])
            if not results:
                print("No more results. Metadata done.")
                break

            last_id, b_obs, b_rows = None, 0, 0
            for obs in results:
                if max_observations is not None and obs_count + b_obs >= max_observations:
                    break
                last_id = obs.get("id", last_id)
                b_obs += 1
                for row in rows_from_obs(obs):
                    writer.writerow(row)
                    b_rows += 1

            out.flush()
            obs_count += b_obs
            row_count += b_rows

            if last_id is None:
                print("Warning: page had no observation IDs. Stopping.")
                break
            id_above = int(last_id)
            print(f"  last_obs_id={id_above}  obs={obs_count:,}  photo_rows={row_count:,}")

            if max_observations is not None and obs_count >= max_observations:
                print(f"Reached max_observations={max_observations}. Stopping.")
                break
            if len(results) < PER_PAGE:
                print("Last page. Metadata done.")
                break
            time.sleep(API_DELAY_SEC)
    finally:
        out.close()

    print(f"Metadata -> {METADATA_CSV}  ({obs_count:,} obs / {row_count:,} photo rows)")
    return METADATA_CSV

In [ ]:
# ============================================================
# Path, mask, RGB, overlay and segment helpers
# ============================================================
def sanitize_token(v) -> str:
    v = re.sub(r"\s+", "_", str(v or "").strip())
    return re.sub(r"[^A-Za-z0-9_]+", "", v)


def genus_species_from_taxon_name(name) -> str:
    if not isinstance(name, str) or not name.strip():
        return "Unknown_unknown"
    parts = name.strip().split()
    if len(parts) >= 2:
        return f"{sanitize_token(parts[0].capitalize())}_{sanitize_token(parts[1].lower())}"
    return sanitize_token(name)


def infer_image_ext_from_url(url) -> str:
    if not isinstance(url, str) or not url:
        return "jpg"
    ext = os.path.splitext(url.split("?", 1)[0])[1].lower().lstrip(".")
    return ext if ext in {"jpg", "jpeg", "png"} else "jpg"


def batch_name_from_global_index(gidx) -> str:
    return f"batch_{(int(gidx) - 1) // BATCH_SIZE + 1:05d}"


def build_paths_for_row(row):
    gidx  = int(row["global_index"])
    batch = batch_name_from_global_index(gidx)
    ext   = infer_image_ext_from_url(str(row["photo_url_original"]))
    image_name = (f"{row['genus_species']}_{int(row['observation_id'])}"
                  f"_image{int(row['image_index'])}.{ext}")
    stem = os.path.splitext(image_name)[0]

    image_dir    = os.path.join(IMAGE_BASE, batch)
    mask_dir     = os.path.join(MASK_BASE, batch)
    overlay_dir  = os.path.join(OVERLAY_BASE, batch)
    segment_dir  = os.path.join(SEGMENT_BASE, batch)
    image_path   = os.path.join(image_dir, image_name)
    overlay_path = os.path.join(overlay_dir, f"{stem}_overlay.png")

    return pd.Series({
        "batch_name": batch, "image_name": image_name, "stem": stem,
        "image_dir": image_dir, "mask_dir": mask_dir,
        "overlay_dir": overlay_dir, "segment_dir": segment_dir,
        "image_path": image_path, "overlay_path": overlay_path,
        "image_relpath": os.path.relpath(image_path, BASE_DIR),
        "mask_batch_relpath": os.path.relpath(mask_dir, BASE_DIR),
    })


def tensor_to_uint8_mask(mask):
    m = mask.detach().cpu().numpy() if isinstance(mask, torch.Tensor) else np.array(mask)
    return (np.squeeze(m) > 0).astype(np.uint8)


def mask_to_bbox(mask):
    ys, xs = np.where(np.squeeze(mask) > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def resize_mask_if_needed(mask, target_size):
    mask = np.squeeze(mask)
    tw, th = target_size
    mh, mw = mask.shape
    if (mw, mh) == (tw, th):
        return mask
    img = Image.fromarray((mask > 0).astype(np.uint8) * 255).resize((tw, th), Image.NEAREST)
    return (np.array(img) > 0).astype(np.uint8)


def sort_instances_by_confidence(masks, scores):
    if not len(masks):
        return [], []
    paired = sorted(zip(masks, scores), key=lambda p: float(p[1]), reverse=True)
    return [p[0] for p in paired], [float(p[1]) for p in paired]


def average_rgb_for_mask(image_rgb_np, mask_np):
    mb = np.squeeze(mask_np) > 0
    if not mb.any():
        return None
    px = image_rgb_np[mb]
    if px.size == 0:
        return None
    a = px.mean(axis=0)
    return tuple(int(round(float(v))) for v in a[:3])


def rgb_tuple_to_string(t):
    return "" if t is None else f"({t[0]},{t[1]},{t[2]})"


def save_segment_png(image_pil, mask_np, out_path) -> bool:
    """Transparent PNG of one segment, cropped to its bounding box."""
    b = mask_to_bbox(mask_np)
    if b is None:
        return False
    x1, y1, x2, y2 = b
    rgba = np.array(image_pil.convert("RGBA"))
    rgba[..., 3] = np.where(np.squeeze(mask_np) > 0, 255, 0).astype(np.uint8)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    Image.fromarray(rgba[y1:y2 + 1, x1:x2 + 1], mode="RGBA").save(out_path)
    return True


def get_font(size=22):
    for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if os.path.exists(p):
            return ImageFont.truetype(p, size=size)
    return ImageFont.load_default()


def draw_label(draw, xy, text, font):
    x, y = xy
    try:
        bb = draw.textbbox((x, y), text, font=font)
        tw, th = bb[2] - bb[0], bb[3] - bb[1]
    except Exception:
        tw, th = draw.textsize(text, font=font)
    draw.rectangle([x, y, x + tw + 2 * TEXT_MARGIN, y + th + 2 * TEXT_MARGIN],
                   fill=(0, 0, 0, 180))
    draw.text((x + TEXT_MARGIN, y + TEXT_MARGIN), text, fill=(255, 255, 255, 255), font=font)


def create_overlay(image_pil, masks_np, scores, output_path) -> int:
    image = image_pil.convert("RGBA")
    w, h = image.size
    mask_layer = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw_layer = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(draw_layer)
    font_size = max(16, int(min(w, h) * 0.025))
    font = get_font(size=font_size)

    valid = 0
    for i, mask in enumerate(masks_np):
        mb = np.squeeze(mask) > 0
        if not mb.any():
            continue
        valid += 1
        alpha = Image.fromarray(mb.astype(np.uint8) * OVERLAY_ALPHA, mode="L")
        color = Image.new("RGBA", image.size, OVERLAY_COLOR + (0,))
        color.putalpha(alpha)
        mask_layer = Image.alpha_composite(mask_layer, color)

        bb = mask_to_bbox(mask)
        if bb is None:
            continue
        x1, y1, x2, y2 = bb
        for off in range(BOX_WIDTH):
            draw.rectangle([x1 - off, y1 - off, x2 + off, y2 + off], outline=BOX_COLOR + (255,))
        label = f"{TEXT_PROMPT} {scores[i]:.3f}" if i < len(scores) else TEXT_PROMPT
        draw_label(draw, (x1, max(0, y1 - font_size - 12)), label, font)

    if valid == 0:
        draw_label(draw, (10, 10), "No detections", font)

    composite = Image.alpha_composite(Image.alpha_composite(image, mask_layer), draw_layer)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    composite.convert("RGB").save(output_path, quality=95)
    return valid

In [ ]:
# ============================================================
# Result logging and the processing dataframe
# ============================================================
CSV_HEADER = [
    "global_index", "batch_name", "image_relpath", "mask_batch_relpath",
    "overlay_relpath", "segment_relpaths", "image_name", "observation_id",
    "image_index", "observed_on", "taxon_id", "taxon_name", "latitude", "longitude",
    "photo_id", "photo_url_original", "num_instances", "mask_files", "mask_scores",
    "avg_RGB", "status", "error", "download_duration_sec", "segmentation_duration_sec",
]


def ensure_csv_header(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(CSV_HEADER)


def append_csv_row(path, row):
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(row)


def load_done_global_indices(path) -> set:
    if not (RESUME and os.path.exists(path)) or os.path.getsize(path) == 0:
        return set()
    try:
        d = pd.read_csv(path, usecols=["global_index", "status"])
        d = d[d["status"].astype(str).isin(["ok", "dl_failed", "seg_failed"])]
        return {int(v) for v in d["global_index"].dropna()}
    except Exception:
        return set()


def log_error(line):
    with open(ERR_LOG, "a", encoding="utf-8") as f:
        f.write(line.strip() + "\n")


def build_processing_dataframe():
    df = pd.read_csv(METADATA_CSV, low_memory=False)
    missing = {"observation_id", "taxon_name", "photo_url_original"} - set(df.columns)
    if missing:
        raise ValueError(f"Metadata CSV missing columns: {missing}")

    df = df[df["photo_url_original"].notna()
            & (df["photo_url_original"].astype(str).str.len() > 0)].copy()
    df["observation_id"] = pd.to_numeric(df["observation_id"], errors="coerce").astype("Int64")
    df = df[df["observation_id"].notna()].copy()

    # Belt-and-braces year filter. The API d1/d2 params already restrict this, but
    # re-checking locally means a stale or hand-edited CSV can never leak other years.
    yr = pd.to_datetime(df["observed_on"], errors="coerce").dt.year
    before = len(df)
    df = df[yr.between(YEAR_MIN, YEAR_MAX)].copy()
    if before != len(df):
        print(f"Year filter {YEAR_MIN}-{YEAR_MAX}: dropped {before - len(df):,} rows")

    df = df.drop_duplicates(subset=["observation_id", "photo_url_original"]).copy()
    df = df.sort_values("observation_id", kind="stable").reset_index(drop=True)
    df["image_index"] = df.groupby("observation_id").cumcount() + 1
    df["genus_species"] = df["taxon_name"].astype(str).apply(genus_species_from_taxon_name)
    df["global_index"] = np.arange(1, len(df) + 1, dtype=int)

    return pd.concat([df, df.apply(build_paths_for_row, axis=1)], axis=1)

In [ ]:
# ============================================================
# Stage 2 - download + SAM3 segmentation
# ============================================================
def run_segmentation():
    df = build_processing_dataframe()
    print(f"Photo rows in range : {len(df):,}")

    ensure_csv_header(OUT_CSV)
    done = load_done_global_indices(OUT_CSV)
    todo = df[~df["global_index"].isin(done)].to_dict(orient="records")
    print(f"Already logged      : {len(done):,}")
    print(f"Remaining to segment: {len(todo):,}")
    if not todo:
        print("Nothing left to do.")
        return

    device  = "cuda" if torch.cuda.is_available() else "cpu"
    t_model = time.perf_counter()
    model     = Sam3Model.from_pretrained("facebook/sam3").to(device).eval()
    processor = Sam3Processor.from_pretrained("facebook/sam3")
    model_load_sec = round(time.perf_counter() - t_model, 2)
    print(f"Model loaded on {device} in {model_load_sec}s")

    session = make_session(max_workers=MAX_WORKERS)

    def download_job(row):
        t0 = time.perf_counter()
        path = str(row["image_path"])
        if SKIP_IF_EXISTS and os.path.exists(path) and os.path.getsize(path) > 0:
            return "exists", None, None, 0.0
        try:
            data = download_image_bytes(session, str(row["photo_url_original"]))
            return "downloaded", data, None, round(time.perf_counter() - t0, 3)
        except Exception as e:
            return "dl_failed", None, str(e), round(time.perf_counter() - t0, 3)

    stats = dict(ok=0, dl_failed=0, seg_failed=0, no_detection=0,
                 skipped_existing=0, total_masks=0)
    t_run    = time.perf_counter()
    gpu_sec  = 0.0
    progress = tqdm(total=len(todo), desc="download + SAM3", dynamic_ncols=True)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        it = iter(todo)
        in_flight = {}
        for _ in range(min(MAX_WORKERS * 2, len(todo))):
            row = next(it, None)
            if row is None:
                break
            in_flight[executor.submit(download_job, row)] = row

        while in_flight:
            future = next(as_completed(in_flight))
            row    = in_flight.pop(future)

            gidx = int(row["global_index"])
            stem = row["stem"]
            image_path, overlay_path = row["image_path"], row["overlay_path"]
            head_row = [gidx, row["batch_name"], row["image_relpath"], row["mask_batch_relpath"]]
            mid_row  = [
                row["image_name"], int(row["observation_id"]), int(row["image_index"]),
                row.get("observed_on", ""), row.get("taxon_id", ""), row["taxon_name"],
                row.get("latitude", ""), row.get("longitude", ""), row.get("photo_id", ""),
                row["photo_url_original"],
            ]
            dl_sec = seg_sec = ""

            try:
                status, data, error, dl_sec = future.result()

                if status == "dl_failed":
                    stats["dl_failed"] += 1
                    append_csv_row(OUT_CSV, head_row + ["", ""] + mid_row +
                                   [0, "", "", "", "dl_failed", error or "", dl_sec, ""])
                    log_error(f"[DL_FAIL] gidx={gidx} url={row['photo_url_original']} err={error}")
                else:
                    if status == "exists":
                        stats["skipped_existing"] += 1
                        image_pil = Image.open(image_path).convert("RGB")
                    else:
                        image_pil = Image.open(BytesIO(data)).convert("RGB")
                        if SAVE_IMAGES:
                            write_bytes_atomic(image_path, data)

                    t_seg  = time.perf_counter()
                    inputs = processor(images=image_pil, text=TEXT_PROMPT,
                                       return_tensors="pt").to(device)
                    with torch.no_grad():
                        outputs = model(**inputs)
                    post = processor.post_process_instance_segmentation(
                        outputs,
                        threshold=CONF_THRESHOLD,
                        mask_threshold=MASK_THRESHOLD,
                        target_sizes=inputs.get("original_sizes").tolist(),
                    )[0]
                    seg_sec  = round(time.perf_counter() - t_seg, 3)
                    gpu_sec += seg_sec

                    masks, scores = post.get("masks"), post.get("scores")
                    if masks is None or scores is None:
                        masks_list, scores_list = [], []
                    else:
                        masks_list = ([masks[i] for i in range(masks.shape[0])]
                                      if isinstance(masks, torch.Tensor) else list(masks))
                        scores_list = (scores.detach().cpu().tolist()
                                       if isinstance(scores, torch.Tensor) else list(scores))
                    masks_list, scores_list = sort_instances_by_confidence(masks_list, scores_list)

                    image_rgb_np = np.array(image_pil)
                    mask_relpaths, seg_relpaths, avg_rgbs, masks_np = [], [], [], []

                    for i, mask in enumerate(masks_list):
                        m = resize_mask_if_needed(tensor_to_uint8_mask(mask), image_pil.size)
                        masks_np.append(m)
                        avg_rgbs.append(average_rgb_for_mask(image_rgb_np, m))

                        if SAVE_MASKS:
                            mp = os.path.join(row["mask_dir"], f"{stem}_instance_{i}.npy")
                            os.makedirs(row["mask_dir"], exist_ok=True)
                            np.save(mp, m)
                            mask_relpaths.append(os.path.relpath(mp, BASE_DIR))

                        if SAVE_SEGMENTS:
                            sp = os.path.join(row["segment_dir"], f"{stem}_segment_{i}.png")
                            if save_segment_png(image_pil, m, sp):
                                seg_relpaths.append(os.path.relpath(sp, BASE_DIR))

                    overlay_relpath = ""
                    if SAVE_OVERLAYS:
                        valid = create_overlay(image_pil, masks_np, scores_list, overlay_path)
                        overlay_relpath = os.path.relpath(overlay_path, BASE_DIR)
                        if valid == 0:
                            stats["no_detection"] += 1
                    elif not masks_np:
                        stats["no_detection"] += 1

                    stats["ok"] += 1
                    stats["total_masks"] += len(masks_list)

                    append_csv_row(OUT_CSV, head_row + [overlay_relpath, ";".join(seg_relpaths)]
                                   + mid_row + [
                        len(masks_list),
                        ";".join(mask_relpaths),
                        ",".join(f"{s:.4f}" for s in scores_list),
                        ",".join(rgb_tuple_to_string(c) for c in avg_rgbs if c is not None),
                        "ok", "", dl_sec, seg_sec,
                    ])

            except Exception as e:
                stats["seg_failed"] += 1
                append_csv_row(OUT_CSV, head_row + ["", ""] + mid_row +
                               [0, "", "", "", "seg_failed", str(e), dl_sec, seg_sec])
                log_error(f"[SEG_FAIL] gidx={gidx} image={image_path} err={e}")

            progress.update(1)
            nxt = next(it, None)
            if nxt is not None:
                in_flight[executor.submit(download_job, nxt)] = nxt

    progress.close()
    wall = round(time.perf_counter() - t_run, 1)

    summary = {
        "taxon_id": TAXON_ID, "prompt": TEXT_PROMPT,
        "years": [YEAR_MIN, YEAR_MAX], "conf_threshold": CONF_THRESHOLD,
        "test_run": TEST_RUN, "device": device,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "images_processed": len(todo), "model_load_sec": model_load_sec,
        "wall_total_sec": wall, "gpu_seg_total_sec": round(gpu_sec, 1),
        "gpu_util_pct": round(100 * gpu_sec / wall, 1) if wall else None,
        "throughput_img_per_s": round(len(todo) / wall, 3) if wall else None,
        **stats,
    }
    with open(TIMING_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n" + json.dumps(summary, indent=2))
    return summary

### Run
Stage 1 pulls metadata, stage 2 downloads and segments. Both resume on re-run, so a Colab disconnect costs only the in-flight image.

In [ ]:
get_metadata(
    TAXON_ID,
    year_min=YEAR_MIN,
    year_max=YEAR_MAX,
    max_observations=TEST_N_OBSERVATIONS if TEST_RUN else None,
)

summary = run_segmentation()

In [ ]:
# ============================================================
# Merge avg_RGB back into the metadata CSV, then report
# ============================================================
def update_metadata_with_avg_rgb():
    md_df = pd.read_csv(METADATA_CSV, low_memory=False)
    s_df  = pd.read_csv(OUT_CSV, low_memory=False)
    s_df  = s_df[s_df["status"].astype(str) == "ok"]

    md_df["observation_id"] = pd.to_numeric(md_df["observation_id"],
                                            errors="coerce").astype("Int64")
    md_df = md_df.sort_values("observation_id", kind="stable").reset_index(drop=True)
    md_df["image_index"] = md_df.groupby("observation_id").cumcount() + 1

    keys = ["observation_id", "image_index", "photo_url_original"]
    s_df = s_df[keys + ["num_instances", "mask_scores", "avg_RGB"]].copy()
    s_df["observation_id"] = pd.to_numeric(s_df["observation_id"],
                                           errors="coerce").astype("Int64")
    s_df["image_index"] = pd.to_numeric(s_df["image_index"], errors="coerce").astype("Int64")

    md_df = md_df.drop(columns=[c for c in ["num_instances", "mask_scores", "avg_RGB"]
                                if c in md_df.columns])
    md_df = md_df.merge(s_df, on=keys, how="left")
    md_df.to_csv(METADATA_CSV, index=False)
    print(f"avg_RGB written for {md_df['avg_RGB'].notna().sum():,} metadata rows")
    return md_df


meta = update_metadata_with_avg_rgb()
res  = pd.read_csv(OUT_CSV, low_memory=False)
ok   = res[res["status"] == "ok"]

print(f"\nOutput folder  : {BASE_DIR}")
print(f"Photos         : {len(res):,}   ok={len(ok):,}  "
      f"dl_failed={(res['status'] == 'dl_failed').sum():,}  "
      f"seg_failed={(res['status'] == 'seg_failed').sum():,}")
if len(ok):
    print(f"With detections: {(ok['num_instances'] > 0).sum():,} "
          f"({100 * (ok['num_instances'] > 0).mean():.1f}%)")
    print(f"Total masks    : {int(ok['num_instances'].sum()):,}")

res.head()

In [ ]:
# ============================================================
# Preview a few overlays (optional)
# ============================================================
import matplotlib.pyplot as plt

hits = ok[ok["num_instances"] > 0].head(6)
if len(hits) == 0:
    print("No detections to preview -- try lowering CONF_THRESHOLD or changing TEXT_PROMPT.")
else:
    n = len(hits)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4.5))
    for ax, (_, r) in zip(np.atleast_1d(axes), hits.iterrows()):
        ax.imshow(Image.open(os.path.join(BASE_DIR, r["overlay_relpath"])))
        ax.set_title(f"{r['num_instances']} x {TEXT_PROMPT}\n{str(r['avg_RGB'])[:28]}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()